# DepositGuard — Model Benchmarking

Trains and compares 4 classifiers (Logistic Regression, Decision Tree, AdaBoost, XGBoost) on `outputs/featured_data.csv` — the original Bank Account Fraud (NeurIPS 2022) features plus the 10 engineered behavioral features from `02_feature_engineering.ipynb` — using a **temporal validation split** and **SMOTE**-resampled training data.

**Leakage caveat carried over from notebook 2:** `employment_risk` is a target-derived ordinal encoding that was fit on the *entire* dataset (all months), including the month 6–7 test period used here. This gives it a slight, known advantage in the benchmark below — in a production pipeline it should be refit on the training fold only.

In [1]:
import time
import numpy as np
import pandas as pd
import joblib
from IPython.display import Markdown, display

from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

OUTPUT_DIR = "../outputs"
RANDOM_STATE = 42

## Load Feature-Engineered Data

Same `float32` / `int8` / `category` dtype optimization used in the previous notebooks.

In [2]:
categorical_cols = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
int8_cols = ["fraud_bool", "month", "rapid_application", "employment_risk", "email_risk", "credit_tier"]

header_cols = pd.read_csv(f"{OUTPUT_DIR}/featured_data.csv", nrows=0).columns
float32_cols = [c for c in header_cols if c not in categorical_cols and c not in int8_cols]

dtype_map = {col: "category" for col in categorical_cols}
dtype_map.update({col: "int8" for col in int8_cols})
dtype_map.update({col: "float32" for col in float32_cols})

df = pd.read_csv(f"{OUTPUT_DIR}/featured_data.csv", dtype=dtype_map)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded: 1,000,000 rows x 42 columns


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,income_ratio,address_stability,identity_score,credit_income_ratio,rapid_application,employment_risk,email_risk,age_income_peer_deviation,credit_tier,fraud_risk_index
0,0,0.3,0.986506,-1.0,25.0,40.0,0.006735,102.453712,AA,1059.0,...,1153.846191,0.0,0.694602,326.086945,1,3,1,-1.112134,2,0.721793
1,0,0.8,0.617426,-1.0,89.0,20.0,0.010095,-0.849551,AD,1658.0,...,833.333374,0.0,0.846970,141.509430,1,4,1,1.027282,2,0.745831
2,0,0.8,0.996707,9.0,14.0,40.0,0.012316,-1.490386,AB,1095.0,...,111.111115,23.0,0.698683,18.867924,1,4,1,0.672656,2,0.706984
3,0,0.6,0.475100,11.0,14.0,30.0,0.006991,-1.863101,AB,3483.0,...,125.000000,25.0,0.490040,24.390242,1,4,1,0.114522,2,0.645842
4,0,0.9,0.842307,-1.0,29.0,40.0,5.742626,47.152496,AA,2339.0,...,105.263161,0.0,0.936923,16.949154,0,4,0,1.029614,2,0.346087


## Feature Set

All original BAF columns plus the 10 engineered features, **excluding** `fraud_bool` (target) and `month` (used only to define the temporal split — including it as a feature would leak the evaluation boundary). The 5 categorical columns are one-hot encoded so the same feature matrix can be fed to all 4 models.

In [3]:
feature_cols = [c for c in df.columns if c not in ["fraud_bool", "month"]]

X = pd.get_dummies(df[feature_cols], columns=categorical_cols, drop_first=True)
y = df["fraud_bool"].astype(int)

print(f"Feature matrix: {X.shape[1]} columns ({len(feature_cols)} raw -> {X.shape[1]} after one-hot encoding)")

Feature matrix: 56 columns (40 raw -> 56 after one-hot encoding)


## Temporal Validation Split

Train on `month <= 5`, test on `month > 5`. The dataset spans months 0–7, with fraud rate climbing from ~0.87% (month 2) to ~1.47% (month 7) as shown in `01_EDA.ipynb` — so the test set (months 6–7) is a genuinely harder, higher-fraud-rate period than training, simulating real-world distribution shift.

In [4]:
train_mask = df["month"] <= 5
test_mask = df["month"] > 5

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {X_train.shape[0]:,} rows (months 0-5), fraud rate {y_train.mean() * 100:.3f}%")
print(f"Test:  {X_test.shape[0]:,} rows (months 6-7), fraud rate {y_test.mean() * 100:.3f}%")

Train: 794,989 rows (months 0-5), fraud rate 1.025%
Test:  205,011 rows (months 6-7), fraud rate 1.404%


## SMOTE Oversampling — Training Data Only

`sampling_strategy=0.1` oversamples the minority (fraud) class up to 10% of the majority class count. Applied strictly to the training split — the test set stays untouched and reflects the real, ~1% fraud rate.

In [5]:
smote = SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {y_train_res.value_counts().to_dict()}")

Before SMOTE: {0: 786838, 1: 8151}
After SMOTE:  {0: 786838, 1: 78683}


## XGBoost `scale_pos_weight` Ratio

Computed as `negative_count / positive_count` on the actual (SMOTE-resampled) training data XGBoost will fit on. Since SMOTE already fixed the minority:majority ratio at `0.1`, this comes out to ~10 regardless of the original imbalance — meaning XGBoost's `scale_pos_weight` and the SMOTE resampling are **both** pushing in the same direction here. That is a deliberately aggressive, compounded imbalance-handling setup (as specified), and combined with `class_weight='balanced'` on the linear/tree models below, we should expect high recall at the cost of precision / false positive rate across the board.

In [6]:
neg, pos = y_train_res.value_counts()[0], y_train_res.value_counts()[1]
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {neg:,} / {pos:,} = {scale_pos_weight:.3f}")

scale_pos_weight = 786,838 / 78,683 = 10.000


## Feature Scaling (Logistic Regression Only)

The tree-based models (Decision Tree, AdaBoost, XGBoost) are scale-invariant, but Logistic Regression benefits from standardized inputs — the raw features span wildly different scales (e.g. `velocity_6h` in the thousands vs. 0/1 flags). A `StandardScaler` fit on the resampled training data is applied only for LR.

In [7]:
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train_res)
X_test_lr = scaler.transform(X_test)

## Train All 4 Models

Trained on the SMOTE-resampled training set (scaled, for Logistic Regression only). **This cell is the slow part of the notebook — XGBoost with 200 trees / depth 6 on ~800K+ rows can take several minutes; the full cell may run 10-20 minutes.**

In [8]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=100, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        scale_pos_weight=scale_pos_weight, n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric="auc", random_state=RANDOM_STATE, n_jobs=-1
    ),
}

fitted_models = {}
train_times = {}

for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()
    if name == "Logistic Regression":
        model.fit(X_train_lr, y_train_res)
    else:
        model.fit(X_train_res, y_train_res)
    elapsed = time.time() - start
    train_times[name] = elapsed
    fitted_models[name] = model
    print(f"  done in {elapsed:.1f}s")

Training Logistic Regression...


C:\Users\onlin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  done in 4.7s
Training Decision Tree...
  done in 16.5s
Training AdaBoost...
  done in 314.4s
Training XGBoost...
  done in 12.8s


## Evaluate on the Untouched Test Set

AUC-ROC, Precision, Recall, F1, and False Positive Rate (`FP / (FP + TN)`), all at the default 0.5 probability threshold.

In [9]:
def evaluate(model, X_eval, y_eval):
    y_proba = model.predict_proba(X_eval)[:, 1]
    y_pred = model.predict(X_eval)

    auc = roc_auc_score(y_eval, y_proba)
    precision = precision_score(y_eval, y_pred, zero_division=0)
    recall = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred).ravel()
    fpr = fp / (fp + tn)

    return {"AUC-ROC": auc, "Precision": precision, "Recall": recall, "F1": f1, "FPR": fpr}

results = {}
for name, model in fitted_models.items():
    X_eval = X_test_lr if name == "Logistic Regression" else X_test
    results[name] = evaluate(model, X_eval, y_test)

## Model Comparison Table

In [10]:
comparison = pd.DataFrame(results).T[["AUC-ROC", "Precision", "Recall", "F1", "FPR"]].round(4)
comparison.index.name = "Model"

training_time = pd.Series(train_times, name="Training Time (s)").round(1)

print("=== Model Comparison ===")
print(comparison.to_string())
print()
print("=== Training Time ===")
print(training_time.to_string())

=== Model Comparison ===
                     AUC-ROC  Precision  Recall      F1     FPR
Model                                                          
Logistic Regression   0.8258     0.1048  0.3846  0.1647  0.0468
Decision Tree         0.7707     0.0653  0.3325  0.1092  0.0677
AdaBoost              0.8570     0.4232  0.0431  0.0782  0.0008
XGBoost               0.8892     0.1984  0.3509  0.2535  0.0202

=== Training Time ===
Logistic Regression      4.7
Decision Tree           16.5
AdaBoost               314.4
XGBoost                 12.8


## Save Comparison Table and XGBoost Model

In [11]:
comparison_path = f"{OUTPUT_DIR}/model_comparison.csv"
comparison.to_csv(comparison_path)
print(f"Saved comparison table to {comparison_path}")

model_path = f"{OUTPUT_DIR}/xgb_model.pkl"
joblib.dump(fitted_models["XGBoost"], model_path)
print(f"Saved XGBoost model to {model_path}")

Saved comparison table to ../outputs/model_comparison.csv
Saved XGBoost model to ../outputs/xgb_model.pkl


## Which Model Won?

In [12]:
winner = comparison["AUC-ROC"].idxmax()
winner_row = comparison.loc[winner]
runner_up = comparison["AUC-ROC"].drop(winner).idxmax()
runner_up_row = comparison.loc[runner_up]

summary = f"""**Winner: `{winner}`**, with AUC-ROC = {winner_row['AUC-ROC']:.4f} (Precision {winner_row['Precision']:.4f}, Recall {winner_row['Recall']:.4f}, F1 {winner_row['F1']:.4f}, FPR {winner_row['FPR']:.4f}).

AUC-ROC is the primary criterion because it's threshold-independent and standard for imbalanced binary classification — it measures ranking quality (how well the model separates fraud from legitimate applications) rather than performance at one arbitrary cutoff, which matters here since the compounded SMOTE + class-weighting setup pushes all 4 models toward high recall / lower precision at the default 0.5 threshold.

Runner-up: `{runner_up}` (AUC-ROC = {runner_up_row['AUC-ROC']:.4f}).

For deployment, the winning model's threshold would still need tuning against a business-defined cost trade-off between missed fraud (false negatives) and blocked legitimate customers (false positives / FPR = {winner_row['FPR']:.4f} at the default threshold)."""

display(Markdown(summary))

**Winner: `XGBoost`**, with AUC-ROC = 0.8892 (Precision 0.1984, Recall 0.3509, F1 0.2535, FPR 0.0202).

AUC-ROC is the primary criterion because it's threshold-independent and standard for imbalanced binary classification — it measures ranking quality (how well the model separates fraud from legitimate applications) rather than performance at one arbitrary cutoff, which matters here since the compounded SMOTE + class-weighting setup pushes all 4 models toward high recall / lower precision at the default 0.5 threshold.

Runner-up: `AdaBoost` (AUC-ROC = 0.8570).

For deployment, the winning model's threshold would still need tuning against a business-defined cost trade-off between missed fraud (false negatives) and blocked legitimate customers (false positives / FPR = 0.0202 at the default threshold).